# Sharing MPLAB ML Projects

✨ **Works in both Google Colab and local Jupyter!** ✨

Learn how to:
- Download complete projects from MPLAB ML
- Package as zip files for sharing
- Extract and upload to accounts
- Rename projects during upload

**Use Cases:**
- 🎓 Training/Hackathons - Share templates with students
- 👥 Team Collaboration - Transfer work between members
- 📦 Project Backup - Keep local copies
- 🔄 Project Templates - Create reusable starting points

---
## 1. Setup

In [ ]:
!pip install mplabml --quiet
print("✓ Dependencies installed")

In [ ]:
import os
import zipfile
from pathlib import Path
from getpass import getpass
from mplabml import Client

print("✓ Libraries imported")

In [ ]:
# Auto-detect environment
try:
    import google.colab
    IN_COLAB = True
    BASE_DIR = '/content'
    print("✓ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    BASE_DIR = os.path.join(os.path.expanduser('~'), 'Downloads')
    print("✓ Running in local Jupyter")

print(f"  Working directory: {BASE_DIR}")
os.makedirs(BASE_DIR, exist_ok=True)

In [ ]:
api_key = getpass("Enter your MPLAB ML API Key: ")
client = Client(api_key=api_key)
print("✓ Connected!")

---
## 2. Select Project

In [ ]:
projects_df = client.list_projects()
print(f"Available Projects: {len(projects_df)}\n")
display(projects_df[['Name', 'Last Modified']])

In [ ]:
PROJECT_TO_SHARE = "Your_Project_Name"  # ← Change this

if PROJECT_TO_SHARE in projects_df['Name'].values:
    client.project = PROJECT_TO_SHARE
    print(f"✓ Selected: {PROJECT_TO_SHARE}")
else:
    print(f"❌ Not found: '{PROJECT_TO_SHARE}'")

---
## 3. Download Project

In [ ]:
print(f"Downloading: {PROJECT_TO_SHARE}...\n")
client.mplabml_project_download()

# Find downloaded project (handles both Colab and local)
search_paths = [
    os.path.join(BASE_DIR, PROJECT_TO_SHARE),
    os.path.join(os.path.expanduser('~'), 'Downloads', PROJECT_TO_SHARE),
    os.path.join('/root/Downloads', PROJECT_TO_SHARE),
    os.path.join(os.getcwd(), PROJECT_TO_SHARE)
]

project_dir = None
for path in search_paths:
    if os.path.exists(path):
        project_dir = path
        break

if not project_dir:
    raise FileNotFoundError(f"Project not found. Searched: {search_paths}")

print(f"✓ Downloaded to: {project_dir}")
print(f"\nContains {len(os.listdir(project_dir))} files")

---
## 4. Create Zip File

In [ ]:
def zip_project(project_path):
    project_path = Path(project_path)
    zip_path = Path(BASE_DIR) / f"{project_path.name}.zip"
    
    print(f"Creating {zip_path.name}...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(project_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, project_path.parent)
                zipf.write(file_path, arcname)
    
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"✓ Created: {zip_path.name} ({size_mb:.2f} MB)")
    return str(zip_path)

zip_path = zip_project(project_dir)

if IN_COLAB:
    print("\n💡 To download: files.download(zip_path) or use file browser")
else:
    print(f"\n📦 Zip saved to: {zip_path}")

---
## 5. Upload Received Zip (Optional)

Skip if continuing from above. Use if starting with a received zip file.

In [ ]:
# Uncomment to upload a zip file
# if IN_COLAB:
#     from google.colab import files
#     uploaded = files.upload()
#     zip_path = os.path.join(BASE_DIR, list(uploaded.keys())[0])
# else:
#     zip_path = input("Enter zip path: ")
# print(f"✓ Using: {zip_path}")

---
## 6. Extract Zip

In [ ]:
def unzip_project(zip_path):
    extract_to = Path(BASE_DIR) / "received_projects"
    extract_to.mkdir(exist_ok=True)
    
    print(f"Extracting {Path(zip_path).name}...")
    with zipfile.ZipFile(zip_path, 'r') as zipf:
        zipf.extractall(extract_to)
        project_name = zipf.namelist()[0].split('/')[0]
    
    project_path = extract_to / project_name
    print(f"✓ Extracted to: {project_path}")
    return str(project_path)

received_project_dir = unzip_project(zip_path)
print("\n📂 Ready for upload!")

---
## 7. Upload (Same Name)

In [ ]:
client.mplabml_project_upload(
    project_name=PROJECT_TO_SHARE,
    project_name_to_upload_as=PROJECT_TO_SHARE,
    project_dir=received_project_dir
)
print(f"✓ Uploaded: {PROJECT_TO_SHARE}")

---
## 8. Upload (New Name)

In [ ]:
NEW_NAME = f"{PROJECT_TO_SHARE}_Shared"

client.mplabml_project_upload(
    project_name=PROJECT_TO_SHARE,
    project_name_to_upload_as=NEW_NAME,
    project_dir=received_project_dir
)
print(f"✓ Uploaded as: {NEW_NAME}")

---
## 9. Verify

In [ ]:
projects_df = client.list_projects()
if NEW_NAME in projects_df['Name'].values:
    print(f"✅ Success! Found: {NEW_NAME}\n")
display(projects_df[['Name', 'Last Modified']])

---
## Summary

**Workflow:**
1. Download → Zip → Share
2. Receive → Extract → Upload

**Works on:**
- ✅ Local Jupyter (~/Downloads)
- ✅ Google Colab (/content)
- ✅ Auto-detects environment

**Shares complete project:**
- Data, labels, segments
- Pipelines, models, queries
- All project configuration

**Remember:** Each user needs their own API key!